In [ ]:
import os
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from spikingjelly.activation_based import neuron, surrogate, functional

from tqdm import tqdm

# Local imports
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model, get_conv_layers
from src.utils.save_utils import load_model
from src.cams.generation_utils import generate_noise_grad_cam
from src.cams.noise_gradcam import SNNNoiseGradCAM
from src.cams.mc_ablation_cam_transient import MCAblationCAMTransient
from src.noisy_node import noise_injection

# Configuration
DATASET_NAME = "dvs_gesture"
DATA_DIR = "./data"
MODEL_NAME = "spiking_vgg11_bn"
MODEL_SAVE_DIR = "./models_save"
N_TIME_BINS = 16
TAU = 2.0
TARGET_LAYER_INDEX = 7  # Index of the layer to analyze
CHANNEL_BATCH_SIZE = 16

# Override k_trials for a deeper convergence check
CONVERGENCE_K = 2000  # Set this higher (e.g., 500) for the paper
CHECK_EVERY = 10

In [ ]:
def get_mc_scores_convergence(model, inputs, class_idx, k_trials, noise_type="surrogate", check_every=10):
    """Compute Monte Carlo smoothed scores and return history."""
    history = {'k': [], 'confidence': [], 'activation': []}
    act_sum, conf_sum = 0.0, 0.0
    
    with noise_injection(model, noise_type=noise_type):
        with torch.no_grad():
            for k in tqdm(range(1, k_trials + 1), desc="MC Score Trials"):
                functional.reset_net(model)
                out = model(inputs)
                logits = out.sum(dim=0)[0]
                conf = F.softmax(logits, dim=0)
                
                act_sum += logits[class_idx].item()
                conf_sum += conf[class_idx].item()
                
                if k % check_every == 0:
                    history['k'].append(k)
                    history['activation'].append(act_sum / k)
                    history['confidence'].append(conf_sum / k)
    
    functional.reset_net(model)
    return history

def calculate_stability(cam_history):
    final_cam = cam_history[max(cam_history.keys())]
    diffs = []
    ks = sorted(cam_history.keys())
    rel_changes = []
    rel_ks = []

    for i, k in enumerate(ks):
        # MSE vs Final
        diff = F.mse_loss(cam_history[k], final_cam).item()
        diffs.append(diff)
        
        # Relative Change vs Previous Checkpoint
        if i > 0:
            prev_k = ks[i-1]
            prev_cam = cam_history[prev_k]
            curr_cam = cam_history[k]
            
            change = torch.norm(curr_cam - prev_cam)
            base = torch.norm(prev_cam) + 1e-8
            rel_change = (change / base).item()
            
            rel_changes.append(rel_change)
            rel_ks.append(k)
            
    return ks, diffs, rel_ks, rel_changes

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running convergence check on {device} with K={CONVERGENCE_K}")

# --- Load Data & Model ---
train_loader, test_loader = get_dataloaders(
    DATASET_NAME, DATA_DIR, batch_size=1, n_time_bins=N_TIME_BINS
)

# Get one sample
events, labels = next(iter(test_loader))
if events.dim() == 4:
    events = events.unsqueeze(1).repeat(1, N_TIME_BINS, 1, 1, 1)
events = events.transpose(0, 1).to(device)

target_class = labels[0].item()
print(f"Target Class: {target_class}")

# Model Setup
sample_events = next(iter(train_loader))[0]
is_single_image = sample_events.dim() == 4
C = sample_events.shape[1] if is_single_image else sample_events.shape[2]
spiking_neuron = neuron.IFNode if is_single_image else neuron.LIFNode

# Determine number of classes
ds = train_loader.dataset
if hasattr(ds, 'classes'): num_classes = len(ds.classes)
elif hasattr(ds, 'num_classes'): num_classes = ds.num_classes
else: num_classes = len(ds.dataset.classes)

model = get_model(
    model_name=MODEL_NAME, num_classes=num_classes, in_channels=C,
    spiking_neuron=spiking_neuron, pretrained=False, detach_reset=False,
    surrogate_function=surrogate.ATan(), tau=TAU if not is_single_image else None
)
functional.set_step_mode(model, 'm')
functional.set_backend(model, "torch", instance=spiking_neuron)
model.to(device)
load_model(model, MODEL_SAVE_DIR, DATASET_NAME)
model.eval()

# --- 1. MC Score Convergence ---
print("Running MC Score Convergence...")
mc_history = get_mc_scores_convergence(
    model, events, target_class, CONVERGENCE_K, noise_type="surrogate", check_every=CHECK_EVERY
)

# --- 2. Noise GradCAM Convergence ---
print("Running Noise GradCAM Convergence...")
target_layer = get_conv_layers(model)[TARGET_LAYER_INDEX]

cam_generator = SNNNoiseGradCAM(
    model, target_layer, k_trials=CONVERGENCE_K, 
    batch_size=CHANNEL_BATCH_SIZE, noise_type="surrogate"
)

cam_history = cam_generator.compute_convergence(
    events, target_class, check_every=CHECK_EVERY
)
grad_ks, grad_diffs, grad_rel_ks, grad_rel_changes = calculate_stability(cam_history)

# --- 3. MC Ablation CAM Convergence ---
print("Running MC Ablation CAM Convergence...")
# Find the LIF layer following the target conv layer
target_lif = None
found_conv = False
for m in model.modules():
    if m is target_layer:
        found_conv = True
    if found_conv and (isinstance(m, neuron.LIFNode) or isinstance(m, neuron.IFNode)):
        target_lif = m
        break

ablation_history = {}
if target_lif is None:
    print("Could not find corresponding LIF layer for Ablation CAM.")
else:
    # Use a smaller K for ablation because it's slow (K * C passes)
    ABLATION_K = 200 
    print(f"Using K={ABLATION_K} for Ablation CAM (computationally expensive)")
    
    ablation_cam = MCAblationCAMTransient(
        model, target_lif, batch_size=CHANNEL_BATCH_SIZE, k_trials=ABLATION_K, noise_type="surrogate"
    )
    ablation_history = ablation_cam.compute_convergence(
        events, target_class, check_every=CHECK_EVERY
    )
    abl_ks, abl_diffs, abl_rel_ks, abl_rel_changes = calculate_stability(ablation_history)


# --- Plotting ---
fig, ax = plt.subplots(1, 4, figsize=(24, 5))

# Plot Confidence
ax[0].plot(mc_history['k'], mc_history['confidence'], marker='o')
ax[0].set_title("MC Confidence Convergence")
ax[0].set_xlabel("K Trials")
ax[0].set_ylabel("Confidence Score")
ax[0].grid(True)

# Plot Activation
ax[1].plot(mc_history['k'], mc_history['activation'], marker='o', color='orange')
ax[1].set_title("MC Logit Convergence")
ax[1].set_xlabel("K Trials")
ax[1].set_ylabel("Logit Sum")
ax[1].grid(True)

# Plot CAM MSE
ax[2].plot(grad_ks[:-1], grad_diffs[:-1], marker='x', color='green', label='Noise GradCAM')
if ablation_history:
    ax[2].plot(abl_ks[:-1], abl_diffs[:-1], marker='o', color='blue', label='Ablation CAM')
ax[2].set_title(f"GradCAM Convergence (MSE vs Final)")
ax[2].set_xlabel("K Trials")
ax[2].set_ylabel("MSE w.r.t Final CAM")
ax[2].set_yscale('log')
ax[2].legend()
ax[2].grid(True)

# Plot Relative Change (Stabilization Metric)
stabilization_threshold = 0.01
ax[3].plot(grad_rel_ks, grad_rel_changes, marker='d', color='purple', label='Noise GradCAM')
if ablation_history:
    ax[3].plot(abl_rel_ks, abl_rel_changes, marker='s', color='cyan', label='Ablation CAM')
ax[3].set_title("Relative Change (Stability Metric)")
ax[3].set_xlabel("K Trials")
ax[3].set_ylabel("||C_k - C_{k-1}|| / ||C_{k-1}||")
ax[3].set_yscale('log')
ax[3].axhline(y=stabilization_threshold, color='r', linestyle='--', label=f'{stabilization_threshold} Threshold')
ax[3].legend()
ax[3].grid(True)
plt.tight_layout()
plt.show()